# Tie-out for notebooks 13–25 (A21)

`10_tie_out` checks `00`–`09`. This notebook gives `13`–`25` the same guarantee and prints, in one place, every
number the paper cites from them. It trains nothing.

Checks: (1) provenance of each result and of the notebooks it depends on; (2) stored hyperparameters equal `02`'s
(and `17`'s Random Forest equals `13`'s); (3) `14` ran on predictions that reproduce the committed `03`/`04`/`06`;
(4) every result records its platform. All checks are collected, the results are saved, and the last cell fails if
any check failed.

Numbered 99 so that it runs last under the README's glob (it was `21_tie_out_revision`).

**Results**: `results/99_tie_out_revision.json`.

In [1]:
import json, os, subprocess, warnings
import nbformat
import pandas as pd
import sybil_pipeline as sp
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
assert not sp.CODE_COMMIT.endswith('-dirty'), f'Run from a clean working tree (CODE_COMMIT={sp.CODE_COMMIT})'
print('Code commit:', sp.CODE_COMMIT)
NBS = [nb for nb in ['13_random_forest_sybil', '14_entity_level_recall', '15_split_tree_report',
                     '16_label_robustness_initial_list', '17_models_10_splits', '18_mimicry_stress',
                     '19_ablation_by_category', '20_temporal_holdout', '22_report_dependence', '23_revision_figures', '24_mimicry_controls', '25_paper_tables'] if os.path.exists(f'results/{nb}.json')]
R = {nb: sp.load_results(nb) for nb in NBS}
print('Results found:', NBS)
CHECKS = []
def check(name, ok, detail=''):
    CHECKS.append(dict(check=name, passed=bool(ok), detail=detail))
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f'  ({detail})' if detail else ''))

Code commit: 4761590
Results found: ['13_random_forest_sybil', '14_entity_level_recall', '15_split_tree_report', '16_label_robustness_initial_list', '17_models_10_splits', '18_mimicry_stress', '19_ablation_by_category', '20_temporal_holdout', '22_report_dependence', '23_revision_figures', '24_mimicry_controls', '25_paper_tables']


## 1. Provenance, with explicit dependencies

In [2]:
DEPS = {'13_random_forest_sybil': ['02_hyperparameter_search', '08_ablation_families'],
        '14_entity_level_recall': ['03_xgboost_sybil', '04_lightgbm_sybil', '06_cross_ensemble_sybil'],
        '15_split_tree_report': ['02_hyperparameter_search', '03_xgboost_sybil', '04_lightgbm_sybil', '08_ablation_families'],
        '16_label_robustness_initial_list': ['02_hyperparameter_search'],
        '17_models_10_splits': ['02_hyperparameter_search', '08_ablation_families', '13_random_forest_sybil'],
        '18_mimicry_stress': ['02_hyperparameter_search', '09_shap_importance'],
        '19_ablation_by_category': ['02_hyperparameter_search', '08_ablation_families'],
        '20_temporal_holdout': ['02_hyperparameter_search'],
        '22_report_dependence': ['02_hyperparameter_search', '15_split_tree_report'],
        '23_revision_figures': ['03_xgboost_sybil', '04_lightgbm_sybil', '09_shap_importance', '11_split_comparison',
                                '16_label_robustness_initial_list', '17_models_10_splits', '18_mimicry_stress',
                                '22_report_dependence'],
        '24_mimicry_controls': ['02_hyperparameter_search', '09_shap_importance', '18_mimicry_stress'],
        '25_paper_tables': ['00_data_pipeline', '02_hyperparameter_search', '07_sensitivity_label_vintage', '08_ablation_families',
                            '11_split_comparison', '13_random_forest_sybil', '14_entity_level_recall', '15_split_tree_report',
                            '16_label_robustness_initial_list', '17_models_10_splits', '19_ablation_by_category',
                            '20_temporal_holdout', '22_report_dependence']}
git = lambda *a: subprocess.run(['git', *a], capture_output=True, text=True)
def code_cells(ref, nb):
    text = git('show', f'{ref}:{nb}.ipynb').stdout
    return [c.source for c in nbformat.reads(text, as_version=4).cells if c.cell_type == 'code'] if text else None

prov = sp.provenance(NBS)
print(prov.to_string(index=False))
for r in prov.itertuples():
    check(f'provenance {r.notebook}', r.dependencies_unchanged, f'commit {r.commit}')
for nb in NBS:
    commit = R[nb]['code_commit']
    for dep in DEPS[nb]:
        same_code = code_cells(commit, dep) == code_cells('HEAD', dep)
        dep_ok = bool(sp.provenance([dep])['dependencies_unchanged'].all())
        check(f'{nb} -> {dep}', same_code and dep_ok,
              f"code of {dep} unchanged since {commit}: {same_code}; {dep}'s own result traces to HEAD: {dep_ok}")

                        notebook  commit  dependencies_unchanged
          13_random_forest_sybil 2adac1c                    True
          14_entity_level_recall 8188899                    True
            15_split_tree_report afd06fd                    True
16_label_robustness_initial_list 7baad94                    True
             17_models_10_splits 2f50786                    True
               18_mimicry_stress 2ac94cb                    True
         19_ablation_by_category 52b3fc4                    True
             20_temporal_holdout dc4d551                    True
            22_report_dependence 1f3da83                    True
             23_revision_figures a2f163c                    True
             24_mimicry_controls 5b83115                    True
                 25_paper_tables 3179d43                    True
PASS  provenance 13_random_forest_sybil  (commit 2adac1c)
PASS  provenance 14_entity_level_recall  (commit 8188899)
PASS  provenance 15_split_tree_report  

PASS  14_entity_level_recall -> 04_lightgbm_sybil  (code of 04_lightgbm_sybil unchanged since 8188899: True; 04_lightgbm_sybil's own result traces to HEAD: True)
PASS  14_entity_level_recall -> 06_cross_ensemble_sybil  (code of 06_cross_ensemble_sybil unchanged since 8188899: True; 06_cross_ensemble_sybil's own result traces to HEAD: True)
PASS  15_split_tree_report -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since afd06fd: True; 02_hyperparameter_search's own result traces to HEAD: True)


PASS  15_split_tree_report -> 03_xgboost_sybil  (code of 03_xgboost_sybil unchanged since afd06fd: True; 03_xgboost_sybil's own result traces to HEAD: True)
PASS  15_split_tree_report -> 04_lightgbm_sybil  (code of 04_lightgbm_sybil unchanged since afd06fd: True; 04_lightgbm_sybil's own result traces to HEAD: True)
PASS  15_split_tree_report -> 08_ablation_families  (code of 08_ablation_families unchanged since afd06fd: True; 08_ablation_families's own result traces to HEAD: True)
PASS  16_label_robustness_initial_list -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 7baad94: True; 02_hyperparameter_search's own result traces to HEAD: True)


PASS  17_models_10_splits -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 2f50786: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  17_models_10_splits -> 08_ablation_families  (code of 08_ablation_families unchanged since 2f50786: True; 08_ablation_families's own result traces to HEAD: True)
PASS  17_models_10_splits -> 13_random_forest_sybil  (code of 13_random_forest_sybil unchanged since 2f50786: True; 13_random_forest_sybil's own result traces to HEAD: True)
PASS  18_mimicry_stress -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 2ac94cb: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  18_mimicry_stress -> 09_shap_importance  (code of 09_shap_importance unchanged since 2ac94cb: True; 09_shap_importance's own result traces to HEAD: True)


PASS  19_ablation_by_category -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 52b3fc4: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  19_ablation_by_category -> 08_ablation_families  (code of 08_ablation_families unchanged since 52b3fc4: True; 08_ablation_families's own result traces to HEAD: True)
PASS  20_temporal_holdout -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since dc4d551: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  22_report_dependence -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 1f3da83: True; 02_hyperparameter_search's own result traces to HEAD: True)
PASS  22_report_dependence -> 15_split_tree_report  (code of 15_split_tree_report unchanged since 1f3da83: True; 15_split_tree_report's own result traces to HEAD: True)


PASS  23_revision_figures -> 03_xgboost_sybil  (code of 03_xgboost_sybil unchanged since a2f163c: True; 03_xgboost_sybil's own result traces to HEAD: True)
PASS  23_revision_figures -> 04_lightgbm_sybil  (code of 04_lightgbm_sybil unchanged since a2f163c: True; 04_lightgbm_sybil's own result traces to HEAD: True)
PASS  23_revision_figures -> 09_shap_importance  (code of 09_shap_importance unchanged since a2f163c: True; 09_shap_importance's own result traces to HEAD: True)
PASS  23_revision_figures -> 11_split_comparison  (code of 11_split_comparison unchanged since a2f163c: True; 11_split_comparison's own result traces to HEAD: True)


PASS  23_revision_figures -> 16_label_robustness_initial_list  (code of 16_label_robustness_initial_list unchanged since a2f163c: True; 16_label_robustness_initial_list's own result traces to HEAD: True)
PASS  23_revision_figures -> 17_models_10_splits  (code of 17_models_10_splits unchanged since a2f163c: True; 17_models_10_splits's own result traces to HEAD: True)
PASS  23_revision_figures -> 18_mimicry_stress  (code of 18_mimicry_stress unchanged since a2f163c: True; 18_mimicry_stress's own result traces to HEAD: True)
PASS  23_revision_figures -> 22_report_dependence  (code of 22_report_dependence unchanged since a2f163c: True; 22_report_dependence's own result traces to HEAD: True)
PASS  24_mimicry_controls -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 5b83115: True; 02_hyperparameter_search's own result traces to HEAD: True)


PASS  24_mimicry_controls -> 09_shap_importance  (code of 09_shap_importance unchanged since 5b83115: True; 09_shap_importance's own result traces to HEAD: True)
PASS  24_mimicry_controls -> 18_mimicry_stress  (code of 18_mimicry_stress unchanged since 5b83115: True; 18_mimicry_stress's own result traces to HEAD: True)
PASS  25_paper_tables -> 00_data_pipeline  (code of 00_data_pipeline unchanged since 3179d43: True; 00_data_pipeline's own result traces to HEAD: True)
PASS  25_paper_tables -> 02_hyperparameter_search  (code of 02_hyperparameter_search unchanged since 3179d43: True; 02_hyperparameter_search's own result traces to HEAD: True)


PASS  25_paper_tables -> 07_sensitivity_label_vintage  (code of 07_sensitivity_label_vintage unchanged since 3179d43: True; 07_sensitivity_label_vintage's own result traces to HEAD: True)
PASS  25_paper_tables -> 08_ablation_families  (code of 08_ablation_families unchanged since 3179d43: True; 08_ablation_families's own result traces to HEAD: True)
PASS  25_paper_tables -> 11_split_comparison  (code of 11_split_comparison unchanged since 3179d43: True; 11_split_comparison's own result traces to HEAD: True)
PASS  25_paper_tables -> 13_random_forest_sybil  (code of 13_random_forest_sybil unchanged since 3179d43: True; 13_random_forest_sybil's own result traces to HEAD: True)
PASS  25_paper_tables -> 14_entity_level_recall  (code of 14_entity_level_recall unchanged since 3179d43: True; 14_entity_level_recall's own result traces to HEAD: True)


PASS  25_paper_tables -> 15_split_tree_report  (code of 15_split_tree_report unchanged since 3179d43: True; 15_split_tree_report's own result traces to HEAD: True)
PASS  25_paper_tables -> 16_label_robustness_initial_list  (code of 16_label_robustness_initial_list unchanged since 3179d43: True; 16_label_robustness_initial_list's own result traces to HEAD: True)
PASS  25_paper_tables -> 17_models_10_splits  (code of 17_models_10_splits unchanged since 3179d43: True; 17_models_10_splits's own result traces to HEAD: True)
PASS  25_paper_tables -> 19_ablation_by_category  (code of 19_ablation_by_category unchanged since 3179d43: True; 19_ablation_by_category's own result traces to HEAD: True)
PASS  25_paper_tables -> 20_temporal_holdout  (code of 20_temporal_holdout unchanged since 3179d43: True; 20_temporal_holdout's own result traces to HEAD: True)
PASS  25_paper_tables -> 22_report_dependence  (code of 22_report_dependence unchanged since 3179d43: True; 22_report_dependence's own result

## 2. Hyperparameters

In [3]:
XGB_P, LGBM_P = sp.load_selected_params()
stored = {'13_random_forest_sybil': ('lgbm_params', LGBM_P), '16_label_robustness_initial_list': ('params', LGBM_P),
          '18_mimicry_stress': ('params', LGBM_P), '19_ablation_by_category': ('params', LGBM_P),
          '20_temporal_holdout': ('params', LGBM_P)}
for nb, (key, ref) in stored.items():
    if nb in R:
        check(f'{nb} LightGBM parameters = 02', R[nb].get(key) == ref, str(R[nb].get(key)))
if '17_models_10_splits' in R:
    p = R['17_models_10_splits']['params']
    check('17 XGBoost parameters = 02', p['xgb'] == XGB_P); check('17 LightGBM parameters = 02', p['lgbm'] == LGBM_P)
    check('17 Random Forest parameters = 13', p['rf'] == R['13_random_forest_sybil']['params'], str(p['rf']))
if '25_paper_tables' in R:
    tn = json.load(open('tables/rev_text_numbers.json'))
    check('25 text numbers: no missing names', not tn['missing'] and not R['25_paper_tables']['text_numbers_missing'],
          f"{len(tn['numbers'])} names; missing {tn['missing']}")
if '25_paper_tables' in R and '16_label_robustness_initial_list' in R:
    # Section 5.3 false-positive rates, recomputed here from results/16 and compared with the named numbers of 25
    tnum = json.load(open('tables/rev_text_numbers.json'))['numbers']
    r16_ = R['16_label_robustness_initial_list']
    fpv_ = [x for x in r16_['fp_composition'] if x['note']][0]
    pt_ = {x['partition']: x for x in r16_['by_partition']}['Test']
    N_, I_, F_, J_ = pt_['negatives'], pt_['negatives_on_list'], fpv_['fp'], fpv_['fp_on_initial_list']
    want_ = {'initial_list_test_negatives': N_, 'initial_list_test_negatives_on_list': I_,
             'fpr_original_labels_numerator': F_, 'fpr_original_labels_denominator': N_, 'fpr_original_labels': F_ / N_,
             'fpr_initial_list_as_positive_numerator': F_ - J_, 'fpr_initial_list_as_positive_denominator': N_ - I_,
             'fpr_initial_list_as_positive': (F_ - J_) / (N_ - I_)}
    ok_ = all(k in tnum and abs(tnum[k]['value'] - v) <= 1e-12 * max(1, abs(v)) for k, v in want_.items())
    check('25 false-positive rates (5.3) = F/N and (F-J)/(N-I) from 16; all I list negatives leave the hypothetical denominator',
          ok_ and N_ - I_ > 0 and 0 <= J_ <= F_,
          f"original {F_}/{N_} = {F_ / N_:.6f}; list as positive {F_ - J_}/{N_ - I_} = {(F_ - J_) / (N_ - I_):.6f}")
if '24_mimicry_controls' in R:
    check('24 LightGBM parameters = 02', R['24_mimicry_controls']['params'] == LGBM_P)
    rc = R['24_mimicry_controls']['regression_check']
    ok = all(abs(v['here'] - v['r18']) < 1e-9 for v in rc['baseline'].values()) and all(abs(x['diff']) < 1e-9 for x in rc['recall'])
    check('24 reproduces 18 (baseline and recall regression checks)', ok, f"{len(rc['recall'])} recall checks")
if '22_report_dependence' in R:
    p = R['22_report_dependence']['params']
    check('22 LightGBM parameters = 02', p['lgbm'] == LGBM_P); check('22 XGBoost parameters = 02', p['xgb'] == XGB_P)
if '15_split_tree_report' in R:
    check('15 uses 02 (loaded at run time; no parameters stored)', 'from 02' in R['15_split_tree_report']['hyperparameters'])

PASS  13_random_forest_sybil LightGBM parameters = 02  ({'num_leaves': 511, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0})
PASS  16_label_robustness_initial_list LightGBM parameters = 02  ({'num_leaves': 511, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0})
PASS  18_mimicry_stress LightGBM parameters = 02  ({'num_leaves': 511, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0})
PASS  19_ablation_by_category LightGBM parameters = 02  ({'num_leaves': 511, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0})
PASS  20_temporal_holdout LightGBM parameters = 02  ({'num_leaves': 511, 'learning_rate': 0.05, 'subsample': 1.0, 'reg_lambda': 1.0})
PASS  17 XGBoost parameters = 02
PASS  17 LightGBM parameters = 02
PASS  17 Random Forest parameters = 13  ({'max_features': 0.3, 'min_samples_leaf': 1, 'max_depth': None, 'max_samples': None, 'n_estimators': 300, 'criterion': 'gini', 'bootstrap': True, 'n_jobs': 4})
PASS  25 text numbers: no missing names  (21

## 3. Inputs of `14`

In [4]:
if '14_entity_level_recall' in R:
    ok = R['14_entity_level_recall']['inputs']['predictions_match_committed_results']
    check('14 predictions reproduce the committed 03/04/06', ok,
          'OK' if ok else 'rerun 03, 04, 06 and then 14 on the machine that produced the committed results (about 15 min)')

PASS  14 predictions reproduce the committed 03/04/06  (OK)


## 4. Platform

In [5]:
plat = {}
for nb in NBS:
    if nb == '25_paper_tables':       # trains nothing; records its own run platform and each source's platform
        check('25 records the platform of each source', 'platform' in R[nb] and bool(R[nb]['platforms']))
        continue
    if nb == '23_revision_figures':   # trains nothing; records each source's platform instead
        check('23 records the platform of each source', all('platform' in s for v in R[nb]['sources'].values() for s in v))
        continue
    p = R[nb].get('platform') or R[nb].get('inputs', {}).get('platform')
    check(f'{nb} records its platform', p is not None)
    if p:
        plat[nb] = {k: p.get(k) for k in ['machine', 'system', 'processor', 'python', 'cpu_count', 'n_jobs']}
plat = pd.DataFrame(plat).T
print(plat.to_string())
machines = set(map(tuple, plat[['machine', 'system']].values))
# 00-12 record no platform; 14 shows their machine differs from arm64/Darwin (03/04 do not reproduce there).
other_00_12 = not R.get('14_entity_level_recall', {}).get('inputs', {}).get('predictions_match_committed_results', True)
print(f'\nPlatforms among 13-25: {sorted(machines)}; 00-12: not recorded'
      + (', and not the platform of 14 (its rerun of 03/04 did not reproduce them).' if other_00_12 else '.'))
MULTI_PLATFORM = len(machines) > 1 or other_00_12
if MULTI_PLATFORM:
    print('WARNING: the paper would cite numbers from more than one platform (00-12 and 13-25). '
          'Comparisons within 13-25 are same-machine; comparisons with 00-12 differ by about 0.002 F1 across platforms.')

PASS  13_random_forest_sybil records its platform
PASS  14_entity_level_recall records its platform
PASS  15_split_tree_report records its platform
PASS  16_label_robustness_initial_list records its platform
PASS  17_models_10_splits records its platform
PASS  18_mimicry_stress records its platform
PASS  19_ablation_by_category records its platform
PASS  20_temporal_holdout records its platform
PASS  22_report_dependence records its platform
PASS  23 records the platform of each source
PASS  24_mimicry_controls records its platform
PASS  25 records the platform of each source
                                 machine system processor   python cpu_count n_jobs
13_random_forest_sybil            x86_64  Linux    x86_64  3.11.15         4      4
14_entity_level_recall            x86_64  Linux    x86_64  3.11.15       NaN    NaN
15_split_tree_report              x86_64  Linux    x86_64  3.11.15         4      4
16_label_robustness_initial_list  x86_64  Linux    x86_64  3.11.15         4     

## 5. Numbers cited from 13–20

In [6]:
CITED = {}
def show(title, obj):
    CITED[title] = obj
    print(f'=== {title}'); print(json.dumps(obj, indent=1, default=float)[:4000]); print()

if '17_models_10_splits' in R:
    show('Model table: 10 group splits, mean ± SD, p vs LightGBM (17)', R['17_models_10_splits']['cite_table'])
if '13_random_forest_sybil' in R:
    r = R['13_random_forest_sybil']
    show('Random Forest (13)', dict(selected=r['params'], test=r['models'][0],
                                     ten_splits={k: v for k, v in r['summary'].items() if k.startswith(('test_f1', 'lgbm_test_f1'))}))
if '14_entity_level_recall' in R:
    t = pd.DataFrame(R['14_entity_level_recall']['table'])
    t = t[(t.threshold_label == 'validation') & (t.bucket == 'all') & t.model.isin(['LightGBM', 'Ensemble'])]
    show('Entity-level recall (14)', t[['entity', 'model', 'entities', 'any_hit', 'majority_hit', 'full_hit',
                                        'residual_allocation_share', 'entities_majority_allocation_unflagged']].to_dict('records'))
if '15_split_tree_report' in R:
    show('Report-level dependence (15)', R['15_split_tree_report']['interpretation_inputs'])
if '16_label_robustness_initial_list' in R:
    r = R['16_label_robustness_initial_list']
    show('Initial-list robustness (16)', dict(counts=r['counts'], fp_at_validation_threshold=[x for x in r['fp_composition'] if x['note']],
                                              table_2x2=r['table_2x2'], variant_B=r['variant_B'],
                                              ten_splits={k: v for k, v in r['summary'].items() if k.startswith('delta_')}))
if '16_label_robustness_initial_list' in R:
    _r = R['16_label_robustness_initial_list']
    _f = [x for x in _r['fp_composition'] if x['note']][0]
    _p = {x['partition']: x for x in _r['by_partition']}['Test']
    show('False-positive rate at the validation threshold (Section 5.3, from 16)',
         dict(threshold=_f['threshold'], N_test_negatives=_p['negatives'], I_test_negatives_on_list=_p['negatives_on_list'],
              F_false_positives=_f['fp'], J_false_positives_on_list=_f['fp_on_initial_list'],
              fpr_original_labels=dict(numerator=_f['fp'], denominator=_p['negatives'], value=_f['fp'] / _p['negatives']),
              fpr_initial_list_as_positive=dict(numerator=_f['fp'] - _f['fp_on_initial_list'],
                                                denominator=_p['negatives'] - _p['negatives_on_list'],
                                                value=(_f['fp'] - _f['fp_on_initial_list']) / (_p['negatives'] - _p['negatives_on_list']))))
if '18_mimicry_stress' in R:
    r = R['18_mimicry_stress']
    show('Mimicry stress test (18)', dict(baseline=r['baseline'], k5_k10=[{k: x[k] for k in ['features', 'attack', 'k', 'recall_mean', 'recall_std']}
                                                                       for x in r['results'] if x['k'] in (5, 10)]))
if '19_ablation_by_category' in R:
    d = pd.DataFrame(R['19_ablation_by_category']['diffs'])
    show('Category ablation (19)', d[d.metric.isin(['f1', 'recall'])][['without', 'category', 'metric', 'diff_mean', 'diff_sd', 'all_higher_n', 'p_corrected']].to_dict('records'))
if '24_mimicry_controls' in R:
    r = R['24_mimicry_controls']
    show('Mimicry controls (24)', dict(shap_shares=r['shap_shares'], single_feature_top5=r['single_feature'][:5],
                                       tie_check=r['single_feature_tie_check']))
if '20_temporal_holdout' in R:
    show('Temporal holdout (20)', R['20_temporal_holdout']['results'])
if '22_report_dependence' in R:
    r = R['22_report_dependence']
    show('Report dependence: seen minus held out (22)', dict(tests=r['tests'], dose_response=r['dose_response'],
                                                            composition=r['composition_summary']))

=== Model table: 10 group splits, mean ± SD, p vs LightGBM (17)
[
 {
  "model": "Logistic regression",
  "F1": "0.2343 \u00b1 0.0037",
  "p F1": "2.32e-13",
  "AP": "0.1626 \u00b1 0.0079",
  "p AP": "6.39e-13",
  "AUROC": "0.8337 \u00b1 0.0026",
  "p AUROC": "2.11e-12"
 },
 {
  "model": "XGBoost",
  "F1": "0.7075 \u00b1 0.0069",
  "p F1": "1",
  "AP": "0.7545 \u00b1 0.0068",
  "p AP": "0.234",
  "AUROC": "0.9688 \u00b1 0.0012",
  "p AUROC": "1"
 },
 {
  "model": "LightGBM",
  "F1": "0.7086 \u00b1 0.0083",
  "p F1": "\u2014",
  "AP": "0.7606 \u00b1 0.0071",
  "p AP": "\u2014",
  "AUROC": "0.9688 \u00b1 0.0019",
  "p AUROC": "\u2014"
 },
 {
  "model": "Random Forest",
  "F1": "0.7023 \u00b1 0.0052",
  "p F1": "1",
  "AP": "0.7537 \u00b1 0.0077",
  "p AP": "0.234",
  "AUROC": "0.9689 \u00b1 0.0014",
  "p AUROC": "1"
 },
 {
  "model": "Ensemble",
  "F1": "0.7100 \u00b1 0.0082",
  "p F1": "1",
  "AP": "0.7627 \u00b1 0.0064",
  "p AP": "0.351",
  "AUROC": "0.9699 \u00b1 0.0016",
  "p AUROC":

## Save, then fail if any check failed

In [7]:
checks = pd.DataFrame(CHECKS)
failed = checks[~checks['passed']]
sp.save_results([], '99_tie_out_revision', extra=dict(checks=CHECKS, all_passed=bool(checks['passed'].all()),
                                                      multiple_platforms_cited=MULTI_PLATFORM,
                                                      platforms=plat.reset_index().to_dict('records'), cited=CITED))
print(f"{int(checks['passed'].sum())} of {len(checks)} checks passed")
print(failed.to_string(index=False) if len(failed) else 'All checks passed')

Saved results/99_tie_out_revision.json
84 of 84 checks passed
All checks passed


In [8]:
assert failed.empty, f'{len(failed)} check(s) failed: ' + '; '.join(failed['check'] + ': ' + failed['detail'])